# DSE Baselines — Notebook 1: PyTorch
MGPred, SDPred, MSSF using **UNIFIED SPLITS**.

**Split A** = Warm-start (pair-wise 10-fold). **Split B** = Drug cold-start (drug-level 10-fold).


In [ ]:
%%time
REPO_URL = 'https://github.com/hungbuile04/DSE.git'
!git clone $REPO_URL
%cd DSE
!pip install -q numpy pandas scikit-learn torch
import os
for split in ['A', 'B']:
    for m in ['MGPred', 'SDPred', 'MSSF']:
        os.makedirs(f'results/{m}_{split}', exist_ok=True)


In [ ]:
SPLIT_TYPE = 'A'  # 'A' = Warm-start (pair-wise), 'B' = Drug cold-start

In [ ]:
import sys
import os
import pickle
import argparse
import numpy as np
sys.path.insert(0, '/content/DSE/shared_data')
from split_adapter import load_splits

---
## 1. MGPred
- `train_test()` returns `(rmse, mae)` and saves best predictions to `final_p{fold}.p`
- Data format: `(SE_id, drug_id, freq)` — reversed order!

In [ ]:
%%time
%cd /content/DSE/MGPred
!sed -i 's/num_workers=16/num_workers=2/g' Ten_Fold_test.py
!sed -i 's/endure_count > 30/endure_count > 5/g' Ten_Fold_test.py

sys.path.insert(0, '/content/DSE/MGPred')
from Ten_Fold_test import train_test, Extract_positive_negative_samples

with open('data/drug_side.pkl', 'rb') as f:
    drug_side = pickle.load(f)

addition_neg, _, final_neg = Extract_positive_negative_samples(drug_side, addition_negative_number='all')
addition_neg = np.vstack((addition_neg, final_neg))
# MGPred format: (se_id, drug_id, freq)
data_neg = [(row[1], row[0], row[2]) for row in addition_neg]

args = argparse.Namespace(
    epochs=20, lr=0.001, embed_dim=64, weight_decay=0.0005,
    N=30000, droprate=0.5, batch_size=256, test_batch_size=256,
    rawpath='/content/DSE/MGPred/data', dataset=''
)

mgpred_results = []
for fold in range(10):
    print(f'\n===== MGPred Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, SPLIT_TYPE)
    # Convert to MGPred format: (se_id, drug_id, freq)
    data_train = [(int(r[1]), int(r[0]), r[2]) for r in train_pos]
    data_test = [(int(r[1]), int(r[0]), r[2]) for r in test_pos]
    
    rmse, mae = train_test(data_train, data_test, data_neg, fold+1, args)
    mgpred_results.append((rmse, mae))
    
    # Load saved best predictions
    with open(f'final_p{fold+1}.p', 'rb') as f:
        ground_i, ground_u, ground_truth, pred = pickle.load(f)
    np.save(f'/content/DSE/results/MGPred_A/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/MGPred_A/fold_{fold}_preds.npy', pred)

print(f'\n=== MGPred Final ===')
print(f'RMSE: {np.mean([r[0] for r in mgpred_results]):.4f} ± {np.std([r[0] for r in mgpred_results]):.4f}')
print(f'MAE:  {np.mean([r[1] for r in mgpred_results]):.4f} ± {np.std([r[1] for r in mgpred_results]):.4f}')

---
## 2. SDPred
- `train_test()` returns only `(auc, aupr, rmse, mae)` — 4 values
- Need to patch return statement to also return predictions
- ConvNCF(7570) is in `up_ten_fold.py`, not `model.py`
- Data format: `(drug_id, se_id, freq)` — normal order

In [ ]:
%%time
%cd /content/DSE
!cp -f shared_data/sdpred_750/* SDPred/data/
%cd /content/DSE/SDPred

# Patch dims: ConvNCF is in up_ten_fold.py line ~150
!sed -i 's/ConvNCF(7570,/ConvNCF(7500,/g' up_ten_fold.py
!sed -i 's/num_workers=16/num_workers=2/g' up_ten_fold.py
!sed -i 's/CUDA_VISIBLE_DEVICES"\] = "2"/CUDA_VISIBLE_DEVICES"] = "0"/g' up_ten_fold.py
!sed -i 's/endure_count > 10/endure_count > 5/g' up_ten_fold.py
!sed -i 's/dropout1=0.8, dropout2=0.8/dropout1=0.5, dropout2=0.5/g' network.py

# CRITICAL: Patch train_test to return predictions
# Original line 198: return i_auc, iPR_auc, rmse, mae
!sed -i 's/return i_auc, iPR_auc, rmse, mae$/return i_auc, iPR_auc, rmse, mae, ground_truth, pred1, pred2/' up_ten_fold.py

sys.path.insert(0, '/content/DSE/SDPred')
from up_ten_fold import train_test as sdpred_train_test
from up_ten_fold import Extract_positive_negative_samples as sdpred_extract

with open('data/drug_side.pkl', 'rb') as f:
    drug_side_sd = pickle.load(f)

add_neg, pos, neg = sdpred_extract(drug_side_sd, 'all')
all_data_neg = [(r[0], r[1], r[2]) for r in np.vstack((add_neg, neg))]
zero_pairs = np.argwhere(drug_side_sd == 0)

args_sd = argparse.Namespace(
    epochs=100, lr=0.0001, embed_dim=32, weight_decay=1e-5,
    N=30000, droprate=0.5, batch_size=128, test_batch_size=128,
    rawpath='/content/DSE/SDPred/data'
)

sdpred_results = []
for fold in range(10):
    print(f'\n===== SDPred Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, SPLIT_TYPE)
    
    np.random.seed(42 + fold)
    shuffled_zeros = np.random.permutation(len(zero_pairs))
    train_neg_idx = shuffled_zeros[:len(train_pos)]
    test_neg_idx = shuffled_zeros[len(train_pos):len(train_pos)+len(test_pos)]
    
    train_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in train_neg_idx]
    test_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in test_neg_idx]
    
    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos] + train_neg
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos] + test_neg
    
    i_auc, iPR_auc, rmse, mae, ground_truth, pred1, pred2 = sdpred_train_test(
        data_train, data_test, all_data_neg, fold+1, args_sd)
    sdpred_results.append((i_auc, iPR_auc, rmse, mae))
    
    np.save(f'/content/DSE/results/SDPred_A/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/SDPred_A/fold_{fold}_preds.npy', pred2)

print(f'\n=== SDPred Final ===')
print(f'AUC:  {np.mean([r[0] for r in sdpred_results]):.4f}')
print(f'AUPR: {np.mean([r[1] for r in sdpred_results]):.4f}')
print(f'RMSE: {np.mean([r[2] for r in sdpred_results]):.4f}')
print(f'MAE:  {np.mean([r[3] for r in sdpred_results]):.4f}')

---
## 3. MSSF
- `train_test(data_train, data_test, args)` — NO fold argument!
- Returns 8 metrics only, need to patch to also return predictions
- Multi-class: class 0-4 → frequency 1-5

In [ ]:
%%time
%cd /content/DSE
!cp -f shared_data/mssf_750/* MSSF/Datas/
%cd /content/DSE/MSSF

# Patch dims: 757 -> 750
!sed -i 's/757\*11/750*11/g' model.py
!sed -i 's/drugs_inputdim=757/drugs_inputdim=750/g' model.py
!sed -i 's/drug_inputdim=757/drug_inputdim=750/g' model.py
!sed -i 's/num_workers=16/num_workers=2/g' mssf.py
!sed -i 's/CUDA_VISIBLE_DEVICES"\] = "3"/CUDA_VISIBLE_DEVICES"] = "0"/g' mssf.py

# CRITICAL: Patch train_test to return raw predictions
# Original: return acc_tested,wf1_tested,maf1_tested,ka_tested,mcc_tested,maprec_tested,mareca_tested,maaupr_tested
# We need rating_te (ground truth) and pred_te (predictions) which exist in the function
!sed -i 's/return acc_tested,wf1_tested,maf1_tested,ka_tested,mcc_tested,maprec_tested,mareca_tested,maaupr_tested/return acc_tested,wf1_tested,maf1_tested,ka_tested,mcc_tested,maprec_tested,mareca_tested,maaupr_tested,rating_te_best,pred_te_best/' mssf.py
# Also need to save best predictions: add tracking variables
!sed -i '/acc_tested = 0/a\    rating_te_best = None\n    pred_te_best = None' mssf.py
!sed -i '/acc_tested = acc_te/a\            rating_te_best = rating_te\n            pred_te_best = pred_te' mssf.py

sys.path.insert(0, '/content/DSE/MSSF')
from mssf import train_test as mssf_train_test

args_mssf = argparse.Namespace(
    epochs=50, lr=0.0001, embed_dim=128, weight_decay=1e-5,
    dropout=0.4, gp=64, batch_size=128, test_batch_size=128,
    rawpath='/content/DSE/MSSF/Datas'
)

mssf_results = []
for fold in range(10):
    print(f'\n===== MSSF Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, SPLIT_TYPE)
    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos]
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos]
    
    # MSSF: train_test(data_train, data_test, args) — NO fold arg
    acc, wf1, maf1, kappa, mcc, prec, recall, aupr, labels, preds = mssf_train_test(
        data_train, data_test, args_mssf)
    mssf_results.append((acc, wf1, maf1, kappa))
    
    # Class 0-4 → frequency 1-5
    np.save(f'/content/DSE/results/MSSF_A/fold_{fold}_labels.npy', np.array(labels) + 1)
    np.save(f'/content/DSE/results/MSSF_A/fold_{fold}_preds.npy', np.array(preds) + 1)

print(f'\n=== MSSF Final ===')
print(f'Acc:   {np.mean([r[0] for r in mssf_results]):.4f}')
print(f'WF1:   {np.mean([r[1] for r in mssf_results]):.4f}')
print(f'MacF1: {np.mean([r[2] for r in mssf_results]):.4f}')
print(f'Kappa: {np.mean([r[3] for r in mssf_results]):.4f}')

---
## Split B: Drug Cold-start

Re-run all 3 models with `SPLIT_TYPE = 'B'`.

> **Note**: SDPred and MSSF use drug-SE profiles as features. In cold-start, test drugs have
> no known associations — the unified split adapter handles this by zeroing out test drug rows
> in the similarity matrices computed inside `read_raw_data()`.


In [ ]:
%%time
# === MGPred Split B ===
%cd /content/DSE/MGPred

mgpred_b_results = []
for fold in range(10):
    print(f'\n===== MGPred Split B Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'B')
    data_train = [(int(r[1]), int(r[0]), r[2]) for r in train_pos]
    data_test = [(int(r[1]), int(r[0]), r[2]) for r in test_pos]
    
    rmse, mae = train_test(data_train, data_test, data_neg, fold+1, args)
    mgpred_b_results.append((rmse, mae))
    
    with open(f'final_p{fold+1}.p', 'rb') as f:
        ground_i, ground_u, ground_truth, pred = pickle.load(f)
    np.save(f'/content/DSE/results/MGPred_B/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/MGPred_B/fold_{fold}_preds.npy', pred)

print(f'\n=== MGPred Split B Final ===')
print(f'RMSE: {np.mean([r[0] for r in mgpred_b_results]):.4f} ± {np.std([r[0] for r in mgpred_b_results]):.4f}')
print(f'MAE:  {np.mean([r[1] for r in mgpred_b_results]):.4f} ± {np.std([r[1] for r in mgpred_b_results]):.4f}')


In [ ]:
%%time
# === SDPred Split B ===
%cd /content/DSE/SDPred

sdpred_b_results = []
for fold in range(10):
    print(f'\n===== SDPred Split B Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'B')
    
    np.random.seed(42 + fold)
    shuffled_zeros = np.random.permutation(len(zero_pairs))
    train_neg_idx = shuffled_zeros[:len(train_pos)]
    test_neg_idx = shuffled_zeros[len(train_pos):len(train_pos)+len(test_pos)]
    
    train_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in train_neg_idx]
    test_neg = [(int(zero_pairs[i][0]), int(zero_pairs[i][1]), 0.0) for i in test_neg_idx]
    
    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos] + train_neg
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos] + test_neg
    
    i_auc, iPR_auc, rmse, mae, ground_truth, pred1, pred2 = sdpred_train_test(
        data_train, data_test, all_data_neg, fold+1, args_sd)
    sdpred_b_results.append((i_auc, iPR_auc, rmse, mae))
    
    np.save(f'/content/DSE/results/SDPred_B/fold_{fold}_labels.npy', ground_truth)
    np.save(f'/content/DSE/results/SDPred_B/fold_{fold}_preds.npy', pred2)

print(f'\n=== SDPred Split B Final ===')
print(f'AUC:  {np.mean([r[0] for r in sdpred_b_results]):.4f}')
print(f'AUPR: {np.mean([r[1] for r in sdpred_b_results]):.4f}')
print(f'RMSE: {np.mean([r[2] for r in sdpred_b_results]):.4f}')
print(f'MAE:  {np.mean([r[3] for r in sdpred_b_results]):.4f}')


In [ ]:
%%time
# === MSSF Split B ===
%cd /content/DSE/MSSF

mssf_b_results = []
for fold in range(10):
    print(f'\n===== MSSF Split B Fold {fold+1}/10 =====')
    train_pos, test_pos = load_splits(fold, 'B')
    data_train = [(int(r[0]), int(r[1]), r[2]) for r in train_pos]
    data_test = [(int(r[0]), int(r[1]), r[2]) for r in test_pos]
    
    acc, wf1, maf1, kappa, mcc, prec, recall, aupr, labels, preds = mssf_train_test(
        data_train, data_test, args_mssf)
    mssf_b_results.append((acc, wf1, maf1, kappa))
    
    np.save(f'/content/DSE/results/MSSF_B/fold_{fold}_labels.npy', np.array(labels) + 1)
    np.save(f'/content/DSE/results/MSSF_B/fold_{fold}_preds.npy', np.array(preds) + 1)

print(f'\n=== MSSF Split B Final ===')
print(f'Acc:   {np.mean([r[0] for r in mssf_b_results]):.4f}')
print(f'WF1:   {np.mean([r[1] for r in mssf_b_results]):.4f}')
print(f'MacF1: {np.mean([r[2] for r in mssf_b_results]):.4f}')
print(f'Kappa: {np.mean([r[3] for r in mssf_b_results]):.4f}')


---
## Unified Evaluation

In [ ]:
%cd /content/DSE
!python shared_data/unified_eval.py --results_dir ./results